# Ion-expert RS-FFF energy and fragmentation diagnostics

This notebook mirrors the energy-correlation side of `eda_rsff_plotting.ipynb` for the ion expert trained at `checkpoints/ion_expert_full`. It expands every hydronium/hydroxide multi-fragmentation frame, plots the fitted interaction-energy components, and then compares the predictions for geometries that have competing decompositions.

The total panel uses interaction energy, `E_total - sum(E_fragment)`, so H2O, H3O+, and HO fragments can live on the same axes without pretending that a single monomer reference applies to all of them.


In [ ]:
%matplotlib inline

import os
import re
import sys
from pathlib import Path

os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")
os.environ.setdefault("MPLCONFIGDIR", "/private/tmp/rsfff_matplotlib")

import matplotlib.pyplot as plt
import numpy as np
import torch

# Resolve the repository root no matter where Jupyter starts the kernel.
ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
os.chdir(ROOT)
for p in (ROOT / "src", ROOT / "scripts"):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

from rsfff.ff.molecular_multipoles import fragment_multipoles, reference_multipoles
from rsfff.ff.multipole import spherical_to_cartesian_quadrupole
from rsfff.ff.units import BOHR_ANG, KJMOL_PER_HARTREE
from rsfff.mlip.heads import env_parameters
from rsfff.mlip.reference_states import AtomicStateReference
from rsfff.train.build_expert import build_expert_model
from rsfff.train.config import load_config
from rsfff.train.data import (
    fragment_view,
    load_cluster_datasets as load_cluster_dataset_expanded,
    load_datasets,
    load_extxyz,
    load_reference_energies,
    split_indices_grouped,
)
from rsfff.train.loss import compute_forces
from rsfff.train.train_eem import resolve_device

plt.rcParams.update({
    "figure.dpi": 140,
    "savefig.dpi": 220,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.color": "#e6e8ef",
    "grid.linewidth": 0.7,
    "axes.labelcolor": "#20242c",
    "axes.edgecolor": "#aeb4c0",
    "xtick.color": "#3c4450",
    "ytick.color": "#3c4450",
    "font.size": 9,
})


In [ ]:
# ---- Regeneration knobs ---------------------------------------------------------------
CHECKPOINT = "checkpoints/ion_expert_full/best.pt"
CONFIG = "configs/ion_expert.yaml"       # fallback only; checkpoint config is preferred
DEVICE = "cpu"                           # CPU is stable for float64 plots; use "auto" if desired
EVAL_SPLIT = "all"                       # "all" or "validation"; validation is grouped by geometry
MAX_FRAMES_PER_SOURCE = None              # e.g. 250 for quick iteration
BATCH_SIZE = 64
COMPETITION_DATASETS = ("w1_h3o+", "w2_h3o+", "w1_oh-", "w2_oh-")
FIGDIR = Path("notebooks/figures")
FIGDIR.mkdir(parents=True, exist_ok=True)
FIGPREFIX = Path(CHECKPOINT).parent.name

# (model term, EDA components summed to make its label, plot title)
ENERGY_TERMS = [
    ("elst", ("cls_elec",), "frozen electrostatics"),
    ("pauli", ("mod_pauli",), "Pauli repulsion"),
    ("disp", ("disp",), "dispersion"),
    ("induction", ("pol", "ct"), "induction (pol + CT)"),
    ("total_interaction", ("energy",), "total interaction"),
]


In [ ]:
def torch_load(path):
    try:
        return torch.load(path, map_location="cpu", weights_only=False)
    except TypeError:
        return torch.load(path, map_location="cpu")


def as_paths(value):
    if value is None:
        return []
    return value if isinstance(value, list) else [value]


def safe_name(value):
    return re.sub(r"[^A-Za-z0-9_.-]+", "_", str(value)).strip("_")


def dataset_tag(path):
    stem = Path(path).stem
    for suffix in ("_wb97mv_tzvpd_pol", "_wb97mv_tzvpd", "_wb97mv", "_tzvpd"):
        if stem.endswith(suffix):
            stem = stem[: -len(suffix)]
    return stem


def stage_of(cfg):
    for stage in cfg.stages:
        if cfg.run_name.endswith(f"_{stage.name}"):
            return stage.name
    return None


def load_expert_checkpoint(checkpoint=CHECKPOINT, device=DEVICE, config_fallback=CONFIG):
    """Rebuild the model from the checkpoint's embedded config, falling back to YAML only for old checkpoints."""
    state = torch_load(checkpoint)
    cfg = state.get("config") or load_config(config_fallback)
    torch.set_default_dtype(torch.float64 if cfg.dtype == "float64" else torch.float32)

    neighbor_types = state.get("neighbor_types")
    if neighbor_types is None:
        probe = load_datasets(cfg.data.path, dtype=torch.get_default_dtype())
        neighbor_types = probe.unique_atomic_numbers
    neighbor_types = tuple(int(z) for z in neighbor_types)

    reference_energies = load_reference_energies(cfg.data.reference_energies, neighbor_types).to(torch.get_default_dtype())
    atomic_states = None
    if cfg.data.atomic_reference_states:
        atomic_states = AtomicStateReference.from_json(
            cfg.data.atomic_reference_states,
            neighbor_types,
            dtype=torch.get_default_dtype(),
        )
    model = build_expert_model(cfg, neighbor_types, reference_energies, atomic_states)
    model.load_state_dict(state["model_state"], strict=True)
    model.eval()
    resolved = torch.device("cpu" if device == "cpu" else resolve_device(device, cfg.dtype))
    model.to(resolved)
    return model, cfg, state, stage_of(cfg), resolved


def load_cluster_datasets_by_file(cfg):
    dtype = torch.float64 if cfg.dtype == "float64" else torch.float32
    out = {}
    for p in as_paths(cfg.data.path):
        out[dataset_tag(p)] = load_cluster_dataset_expanded(
            [p], dtype=dtype, fragmentations=cfg.data.fragmentations
        )
    return out


def frame_indices(dataset, cfg):
    n = len(dataset)
    if EVAL_SPLIT == "validation":
        if getattr(dataset, "_group_id", None) is not None:
            _, val = split_indices_grouped(dataset._group_id, cfg.data.holdout_fraction, cfg.data.seed)
            idx = np.sort(val.cpu().numpy())
        else:
            rng = np.random.default_rng(cfg.data.seed)
            perm = rng.permutation(n)
            n_val = max(1, int(round(n * cfg.data.holdout_fraction)))
            idx = np.sort(perm[:n_val])
    elif EVAL_SPLIT == "all":
        idx = np.arange(n)
    else:
        raise ValueError("EVAL_SPLIT must be 'all' or 'validation'")
    if MAX_FRAMES_PER_SOURCE is not None:
        idx = idx[:MAX_FRAMES_PER_SOURCE]
    return idx.tolist()


def to_np(x):
    return x.detach().cpu().numpy()


def fragment_to_system(batch):
    if batch.fragment_to_batch is not None:
        return batch.fragment_to_batch
    f2b = batch.batch_idx.new_zeros(batch.n_fragments)
    return f2b.scatter_(0, batch.fragment_idx, batch.batch_idx)


def pool_fragments_to_frames(values, batch):
    f2b = fragment_to_system(batch)
    return values.new_zeros(batch.n_systems).index_add_(0, f2b, values)


def fragments_per_frame(batch):
    f2b = fragment_to_system(batch)
    return torch.bincount(f2b, minlength=batch.n_systems).to(batch.positions.dtype)


def pooled_applicability(out, batch):
    scores = getattr(out, "applicability", None)
    if scores is None:
        return None
    f2b = fragment_to_system(batch)
    total = scores.new_zeros(batch.n_systems).index_add_(0, f2b, scores)
    count = scores.new_zeros(batch.n_systems).index_add_(0, f2b, torch.ones_like(scores))
    return total / count.clamp(min=1.0)


def metrics(ref, pred):
    ref = np.asarray(ref)
    pred = np.asarray(pred)
    err = pred - ref
    if len(ref) > 1 and np.std(ref) > 0 and np.std(pred) > 0:
        r2 = float(np.corrcoef(ref, pred)[0, 1] ** 2)
    else:
        r2 = np.nan
    return float(np.abs(err).mean()), float(np.sqrt(np.mean(err ** 2))), r2


PALETTE = {
    "w1_h3o+": "#276ef1",
    "w2_h3o+": "#0f8b8d",
    "w1_oh-": "#d64545",
    "w2_oh-": "#db6d00",
    "w2": "#6f58c9",
    "w3": "#1b998b",
    "w4": "#b05c00",
    "w5": "#7a4cc2",
    "h2o": "#276ef1",
    "h2o_opt": "#1b998b",
    "h3o+": "#0f8b8d",
    "h3o+_opt": "#47a9cf",
    "oh-": "#d64545",
    "oh-_opt": "#db6d00",
}


In [ ]:
model, cfg, state, stage_name, device = load_expert_checkpoint()
datasets = load_cluster_datasets_by_file(cfg)

n_all = sum(p.numel() for p in model.parameters())
n_env = sum(p.numel() for _n, p in env_parameters(model))

print(f"checkpoint: {CHECKPOINT}")
print(f"stage: {stage_name or 'single'}    run_name: {cfg.run_name}")
print(f"epoch: {state.get('epoch', 'unknown')}    val_loss: {state.get('val_loss', float('nan')):.6g}")
print(f"device: {device}    dtype: {cfg.dtype}")
print(f"experts: {sorted(model.experts.experts)}   induction: {model.induction}")
print(f"parameters: {n_all}, of which {n_env} ({100 * n_env / n_all:.1f}%) in the environment slot")
print(f"env_penalty_weight: {cfg.expert.env_penalty_weight}")
print("datasets:", {tag: len(ds) for tag, ds in datasets.items()})
print("contested groups:", {
    tag: int(sum(c > 1 for c in torch.bincount(ds._group_id).tolist()))
    for tag, ds in datasets.items()
    if getattr(ds, "_group_id", None) is not None
})


## Energy Component Correlations

These are the cluster-level labels the expert loss fits: frozen electrostatics, modified Pauli repulsion, dispersion, induction as `pol + ct`, and the total interaction energy after subtracting the one-body fragment sum.


In [ ]:
def collect_energy_predictions(model, datasets, cfg, device, batch_size=BATCH_SIZE):
    rows = {term: {"ref": [], "pred": [], "tag": []} for term, _, _ in ENERGY_TERMS}
    for tag, ds in datasets.items():
        indices = frame_indices(ds, cfg)
        for start in range(0, len(indices), batch_size):
            batch = ds.flat_batch(indices[start:start + batch_size]).to(device)
            with torch.no_grad():
                out = model(batch)
            for term, target, _label in ENERGY_TERMS:
                if term == "total_interaction":
                    if batch.fragment_energy is None:
                        continue
                    pred = out.energy - pool_fragments_to_frames(out.fragment_energy, batch)
                    ref = batch.energy - pool_fragments_to_frames(batch.fragment_energy, batch)
                else:
                    if term not in out.interaction or batch.eda is None:
                        continue
                    if any(k not in batch.eda for k in target):
                        continue
                    pred = out.interaction[term]
                    ref = sum(batch.eda[k] for k in target)
                rows[term]["pred"].append(to_np(pred) * KJMOL_PER_HARTREE)
                rows[term]["ref"].append(to_np(ref) * KJMOL_PER_HARTREE)
                rows[term]["tag"].extend([tag] * len(pred))
    packed = {}
    for term, data in rows.items():
        if data["pred"]:
            packed[term] = {
                "pred": np.concatenate(data["pred"]),
                "ref": np.concatenate(data["ref"]),
                "tag": np.asarray(data["tag"]),
            }
    return packed


energy = collect_energy_predictions(model, datasets, cfg, device)
print(f"{'term':<20}{'n':>8}{'MAE':>12}{'RMSE':>12}{'R2':>10}")
for term, _target, label in ENERGY_TERMS:
    if term not in energy:
        continue
    mae, rmse, r2 = metrics(energy[term]["ref"], energy[term]["pred"])
    print(f"{term:<20}{len(energy[term]['ref']):>8}{mae:>12.4f}{rmse:>12.4f}{r2:>10.5f}  {label}")


In [ ]:
def plot_parity_grid(data, labels, *, title, unit, path=None, ncols=3):
    keys = [k for k, _ in labels if k in data]
    nrows = int(np.ceil(len(keys) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(4.1 * ncols, 3.6 * nrows), squeeze=False)
    for ax in axes.ravel():
        ax.set_visible(False)
    for ax, key in zip(axes.ravel(), keys):
        ax.set_visible(True)
        d = data[key]
        ref, pred, tags = d["ref"], d["pred"], d.get("tag")
        lo = min(np.min(ref), np.min(pred))
        hi = max(np.max(ref), np.max(pred))
        pad = 0.04 * (hi - lo if hi > lo else 1.0)
        ax.plot([lo - pad, hi + pad], [lo - pad, hi + pad], color="#242934", lw=1.0, ls=(0, (4, 3)))
        for tag in sorted(set(tags)):
            m = tags == tag
            ax.scatter(ref[m], pred[m], s=8, alpha=0.58, lw=0, color=PALETTE.get(tag, "#555"), label=tag)
        mae, rmse, r2 = metrics(ref, pred)
        ax.text(0.04, 0.96, f"MAE {mae:.3g}\nRMSE {rmse:.3g}\nR2 {r2:.4f}",
                transform=ax.transAxes, ha="left", va="top", fontsize=8,
                bbox={"boxstyle": "round,pad=0.25", "fc": "white", "ec": "#d6dae3", "alpha": 0.85})
        ax.set_title(dict(labels)[key])
        ax.set_xlabel(f"reference ({unit})")
        ax.set_ylabel(f"predicted ({unit})")
        ax.set_xlim(lo - pad, hi + pad)
        ax.set_ylim(lo - pad, hi + pad)
        ax.legend(frameon=False, fontsize=7, markerscale=1.5)
    fig.suptitle(title, y=1.02, fontsize=13)
    fig.tight_layout()
    if path:
        fig.savefig(path, bbox_inches="tight")
    return fig


energy_labels = [(term, label) for term, _target, label in ENERGY_TERMS]
fig = plot_parity_grid(
    energy,
    energy_labels,
    title="Ion expert interaction-energy component correlations",
    unit="kJ/mol",
    path=FIGDIR / f"{FIGPREFIX}_ion_energy_correlations.png",
)
plt.show()


## Competing Fragmentations

The hydronium and hydroxide microsolvation files carry two or three decompositions of the same nuclei. The plots below compare those alternatives inside each geometry. The reference-best alternative is defined as the one with the smallest `|eda_pol + eda_ct|`, matching the training target for the applicability head.


In [ ]:
def collect_competing_fragmentations(model, datasets, cfg, device, batch_size=BATCH_SIZE):
    rows = []
    for tag, ds in datasets.items():
        if tag not in COMPETITION_DATASETS or getattr(ds, "_group_id", None) is None:
            continue
        indices = frame_indices(ds, cfg)
        for start in range(0, len(indices), batch_size):
            local_indices = indices[start:start + batch_size]
            batch = ds.flat_batch(local_indices).to(device)
            if batch.group_id is None or batch.eda is None:
                continue
            with torch.no_grad():
                out = model(batch)
            if batch.fragment_energy is None:
                continue
            ref_total = batch.energy - pool_fragments_to_frames(batch.fragment_energy, batch)
            pred_total = out.energy - pool_fragments_to_frames(out.fragment_energy, batch)
            ref_ind = batch.eda["pol"] + batch.eda["ct"] if "pol" in batch.eda and "ct" in batch.eda else None
            pred_ind = out.interaction.get("induction")
            app = pooled_applicability(out, batch)
            for i, frame_index in enumerate(local_indices):
                rows.append({
                    "tag": tag,
                    "frame_index": int(frame_index),
                    "group": int(batch.group_id[i].detach().cpu()),
                    "ref_total": float(ref_total[i].detach().cpu()) * KJMOL_PER_HARTREE,
                    "pred_total": float(pred_total[i].detach().cpu()) * KJMOL_PER_HARTREE,
                    "ref_induction": float(ref_ind[i].detach().cpu()) * KJMOL_PER_HARTREE if ref_ind is not None else np.nan,
                    "pred_induction": float(pred_ind[i].detach().cpu()) * KJMOL_PER_HARTREE if pred_ind is not None else np.nan,
                    "applicability": float(app[i].detach().cpu()) if app is not None else np.nan,
                })

    by_group = {}
    for i, row in enumerate(rows):
        by_group.setdefault((row["tag"], row["group"]), []).append(i)
    for idxs in by_group.values():
        idxs.sort(key=lambda i: rows[i]["frame_index"])
        for alt, i in enumerate(idxs):
            rows[i]["alternative"] = alt
        ref_abs = np.array([abs(rows[i]["ref_induction"]) for i in idxs])
        pred_abs = np.array([abs(rows[i]["pred_induction"]) for i in idxs])
        best = idxs[int(np.nanargmin(ref_abs))]
        pred_best = idxs[int(np.nanargmin(pred_abs))]
        app_values = np.array([rows[i]["applicability"] for i in idxs])
        app_best = None if np.isnan(app_values).all() else idxs[int(np.nanargmax(app_values))]
        pred_spread = max(rows[i]["pred_total"] for i in idxs) - min(rows[i]["pred_total"] for i in idxs)
        for i in idxs:
            rows[i]["is_ref_best"] = i == best
            rows[i]["is_pred_induction_best"] = i == pred_best
            rows[i]["is_app_best"] = (app_best is not None and i == app_best)
            rows[i]["d_ref_perturbation"] = abs(rows[i]["ref_induction"]) - abs(rows[best]["ref_induction"])
            rows[i]["d_pred_perturbation_vs_ref_best"] = abs(rows[i]["pred_induction"]) - abs(rows[best]["pred_induction"])
            rows[i]["d_pred_total_vs_ref_best"] = rows[i]["pred_total"] - rows[best]["pred_total"]
            rows[i]["pred_total_spread"] = pred_spread
    return rows


competition = collect_competing_fragmentations(model, datasets, cfg, device)
by_group = {}
for row in competition:
    by_group.setdefault((row["tag"], row["group"]), []).append(row)

print(f"contested alternatives: {len(competition)} across {len(by_group)} geometries")
for tag in COMPETITION_DATASETS:
    groups = [rows for (t, _g), rows in by_group.items() if t == tag and len(rows) > 1]
    if not groups:
        continue
    ind_hits = sum(any(r["is_ref_best"] and r["is_pred_induction_best"] for r in rows) for rows in groups)
    app_hits = sum(any(r["is_ref_best"] and r["is_app_best"] for r in rows) for rows in groups)
    spreads = np.array([rows[0]["pred_total_spread"] for rows in groups])
    print(
        f"{tag:<8} groups={len(groups):>4}  induction-rank acc={ind_hits / len(groups):.3f}  "
        f"app acc={app_hits / len(groups):.3f}  median total spread={np.median(spreads):.3f} kJ/mol"
    )


In [ ]:
def plot_competing_fragmentations(rows, path=None):
    if not rows:
        raise ValueError("no competing-fragmentation rows were collected")
    tags = [t for t in COMPETITION_DATASETS if any(r["tag"] == t for r in rows)]
    fig, axes = plt.subplots(1, 3, figsize=(14.3, 4.2))

    ax = axes[0]
    for tag in tags:
        subset = [r for r in rows if r["tag"] == tag and not r["is_ref_best"]]
        if not subset:
            continue
        x = np.array([r["d_ref_perturbation"] for r in subset])
        y = np.array([r["d_pred_perturbation_vs_ref_best"] for r in subset])
        ax.scatter(x, y, s=12, alpha=0.62, lw=0, color=PALETTE.get(tag, "#555"), label=tag)
    lim = ax.get_xlim()
    ax.axhline(0.0, color="#242934", lw=1.0, ls=(0, (4, 3)))
    ax.set_xlim(left=min(0, lim[0]))
    ax.set_xlabel("reference penalty vs best |pol + CT| (kJ/mol)")
    ax.set_ylabel("predicted penalty vs ref-best |induction| (kJ/mol)")
    ax.set_title("(a) does the energy favor the same fragmentation?")
    ax.legend(frameon=False, fontsize=7)

    ax = axes[1]
    for tag in tags:
        subset = [r for r in rows if r["tag"] == tag]
        x = np.array([r["ref_total"] for r in subset])
        y = np.array([r["pred_total"] for r in subset])
        alt = np.array([r["alternative"] for r in subset])
        for a in sorted(set(alt)):
            m = alt == a
            marker = ["o", "s", "^"][a % 3]
            ax.scatter(x[m], y[m], s=11, alpha=0.5, marker=marker, lw=0,
                       color=PALETTE.get(tag, "#555"), label=f"{tag} alt {a}")
    lo = min(min(r["ref_total"] for r in rows), min(r["pred_total"] for r in rows))
    hi = max(max(r["ref_total"] for r in rows), max(r["pred_total"] for r in rows))
    pad = 0.04 * (hi - lo if hi > lo else 1.0)
    ax.plot([lo - pad, hi + pad], [lo - pad, hi + pad], color="#242934", lw=1.0, ls=(0, (4, 3)))
    ax.set_xlim(lo - pad, hi + pad)
    ax.set_ylim(lo - pad, hi + pad)
    ax.set_xlabel("reference total interaction (kJ/mol)")
    ax.set_ylabel("predicted total interaction (kJ/mol)")
    ax.set_title("(b) total energy for each alternative")
    ax.legend(frameon=False, fontsize=6, ncol=2)

    ax = axes[2]
    positions = np.arange(len(tags))
    values = []
    for tag in tags:
        seen = {}
        for r in rows:
            if r["tag"] == tag:
                seen[(r["tag"], r["group"])] = r["pred_total_spread"]
        values.append(np.array(list(seen.values())))
    ax.boxplot(values, positions=positions, widths=0.55, patch_artist=True,
               boxprops={"facecolor": "#e9edf5", "edgecolor": "#6b7280"},
               medianprops={"color": "#242934"}, whiskerprops={"color": "#6b7280"},
               capprops={"color": "#6b7280"}, flierprops={"marker": ".", "markersize": 3, "alpha": 0.35})
    ax.set_xticks(positions)
    ax.set_xticklabels(tags, rotation=20, ha="right")
    ax.set_ylabel("max-min predicted total interaction (kJ/mol)")
    ax.set_title("(c) prediction spread for one geometry")

    fig.suptitle("Competing hydronium/hydroxide-water fragmentations", y=1.03, fontsize=13)
    fig.tight_layout()
    if path:
        fig.savefig(path, bbox_inches="tight")
    return fig


if competition:
    fig = plot_competing_fragmentations(
        competition,
        path=FIGDIR / f"{FIGPREFIX}_competing_fragmentations.png",
    )
    plt.show()


## Notes

- `total interaction` is computed as `E_total - sum(E_fragment)` on both the model and reference side. That is more useful for the ion corpus than subtracting a single H2O monomer reference, because these systems mix H2O, H3O+, and HO fragments.
- The contested-fragmentation panels use the hydronium/hydroxide files named in `COMPETITION_DATASETS`. Within each geometry, the reference-best decomposition is the one with the smallest `|eda_pol + eda_ct|`, matching the applicability target used during training.
- Panel (c) is intentionally not a parity plot: the same nuclei have one total energy, so any spread across alternative decompositions is model dependence on the chosen fragmentation.
